In [ ]:
# rm -rf ../tmp/prostt5_weights ../tmp/foldseek_tmp
# mkdir -p ../tmp/foldseek_tmp

# foldseek databases ProstT5 ../tmp/prostt5_weights ../tmp/foldseek_tmp
# foldseek createdb ../sarg_ref.fa ../tmp/targetDB --prostt5-model ../tmp/prostt5_weights

In [ ]:
# foldseek easy-search \
#   ../sarg_ref.fa \
#   ../tmp/targetDB \
#   ../tmp/foldseek.tsv \
#   foldseek_tmp \
#   --prostt5-model ../tmp/prostt5_weights \
#   --sort-by-structure-bits 0 \
#   -e 10 \
#   --format-output "query,target,evalue,bits,fident,alnlen,qlen,qstart,qend,tlen,tstart,tend"

In [ ]:
a = []
with open('../tmp/foldseek.tsv') as f:
    for line in f:
        ls = line.rstrip().split('\t')
        if ls[0] != ls[1]:
            a.append(ls)

In [ ]:
ref = set()
with open('../sarg_ref.fa') as f:
    for line in f:
        if line[0] == '>':
            ref.add(line[1:].split()[0])

In [ ]:
import pandas as pd
source = pd.read_table('../misc/summary.tsv').set_index('sarg').source.to_dict()
df = pd.DataFrame(a)
df['qcov'] = (df[8].astype(int) - df[7].astype(int) + 1) / df[6].astype(int)
df['evalue'] = df[2].astype(float)
df['source'] = df[0].map(source)
df = df[(df.qcov > 0.75) & (df.evalue < 1)][[0, 1, 3, 4, 'qcov', 'evalue', 'source']]

In [ ]:
# dff = df.groupby(0).first()
# dff[dff.index.str.split('|').str.get(1) != dff[1].str.split('|').str.get(1)].head(50)

In [ ]:
b = df.groupby(0).size().sort_values()
{x for x,y in source.items() if 'REF' in y and x in (ref - set(b.index))}